# 📖 Notebook 2: Portfolio & Position Tracking

When a user buys or sells stock, their **portfolio** needs to be updated in real time.
This notebook covers how a brokerage tracks positions, calculates profit/loss, and
keeps everything consistent.

## Learning Objectives

By the end of this notebook you will understand:
- How **positions** track what a user owns
- How trades update positions and cash balances
- How to calculate **unrealized P&L** (profit/loss on open positions)
- How to calculate **realized P&L** (profit/loss from completed sales)
- Why we use a **positions table** instead of recalculating from trades every time

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/robinhood
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM positions")
    print(f"✅ PostgreSQL connected — {cur.fetchone()[0]} positions loaded")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 📚 What Is a Position?

A **position** represents how many shares of a stock a user currently holds,
and at what average cost they bought them.

```
  Alice's Portfolio
  ┌────────┬──────────┬──────────────┬──────────────┐
  │ Symbol │ Quantity │   Avg Cost   │ Market Value │
  ├────────┼──────────┼──────────────┼──────────────┤
  │ AAPL   │    50    │   $180.00    │   $191.50    │
  │ META   │    20    │   $500.00    │   $522.10    │
  └────────┴──────────┴──────────────┴──────────────┘
                        ↑ what she paid   ↑ current price
```

### Why a Positions Table?

We *could* calculate positions by scanning all trades every time, but:
- A user with 10,000 trades would need 10,000 rows scanned for every portfolio view
- This gets slower as the user trades more

Instead, we keep a **materialized position** — one row per (user, symbol) — and
update it whenever a trade executes. This is an O(1) lookup vs O(n) scan.

In [ ]:
# Let's look at Alice's current positions (from our seed data)

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT s.ticker, p.quantity, 
           p.avg_cost_cents / 100.0 AS avg_cost,
           s.last_price_cents / 100.0 AS current_price,
           (s.last_price_cents - p.avg_cost_cents) * p.quantity / 100.0 AS unrealized_pnl
    FROM positions p
    JOIN symbols s ON p.symbol_id = s.id
    WHERE p.user_id = 1
    ORDER BY s.ticker
""")

positions = cur.fetchall()

print("📊 Alice's Portfolio")
print("=" * 70)
print(f"{'Ticker':<8} {'Qty':>5} {'Avg Cost':>10} {'Mkt Price':>10} {'P&L':>12}")
print("-" * 70)

total_value = 0
total_cost = 0
for pos in positions:
    pnl = float(pos['unrealized_pnl'])
    pnl_str = f"{'🟢' if pnl >= 0 else '🔴'} ${abs(pnl):,.2f}"
    print(f"{pos['ticker']:<8} {pos['quantity']:>5} ${float(pos['avg_cost']):>9,.2f} "
          f"${float(pos['current_price']):>9,.2f} {pnl_str:>12}")
    total_value += float(pos['current_price']) * pos['quantity']
    total_cost += float(pos['avg_cost']) * pos['quantity']

print("-" * 70)
total_pnl = total_value - total_cost
print(f"{'Total':<8} {'':>5} ${total_cost:>9,.2f} ${total_value:>9,.2f} "
      f"{'🟢' if total_pnl >= 0 else '🔴'} ${abs(total_pnl):>9,.2f}")

conn.close()

## 🔄 Updating Positions When Trades Happen

When a BUY trade executes, we need to:
1. Increase the position quantity
2. Update the average cost
3. Deduct cash from the user's balance

When a SELL trade executes, we need to:
1. Decrease the position quantity
2. Keep average cost the same (it's the cost of shares we still hold)
3. Add cash to the user's balance
4. Calculate realized P&L

### Average Cost Calculation

When buying more shares of something you already own:

```
new_avg_cost = (old_qty × old_avg_cost + new_qty × new_price) / (old_qty + new_qty)
```

Example: Alice has 50 AAPL at $180. She buys 10 more at $191.50:
```
new_avg_cost = (50 × $180 + 10 × $191.50) / (50 + 10) = $181.92
```

In [ ]:
def process_trade(user_id: int, ticker: str, side: str,
                  quantity: int, price_cents: int):
    """
    Process a filled trade: update position, balance, and record the trade.
    This runs inside a single transaction for consistency.
    """
    conn = psycopg2.connect(**DB_CONFIG)
    # Use a transaction (no autocommit) for atomicity
    conn.autocommit = False
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Look up symbol
        cur.execute("SELECT id FROM symbols WHERE ticker = %s", (ticker,))
        symbol = cur.fetchone()
        symbol_id = symbol['id']

        total_cost_cents = price_cents * quantity

        if side == 'buy':
            # Deduct cash
            cur.execute("""
                UPDATE users SET balance_cents = balance_cents - %s
                WHERE id = %s AND balance_cents >= %s
                RETURNING balance_cents
            """, (total_cost_cents, user_id, total_cost_cents))
            result = cur.fetchone()
            if not result:
                raise ValueError("Insufficient funds!")

            # Update or create position
            cur.execute("""
                SELECT quantity, avg_cost_cents FROM positions
                WHERE user_id = %s AND symbol_id = %s
            """, (user_id, symbol_id))
            existing = cur.fetchone()

            if existing:
                old_qty = existing['quantity']
                old_cost = existing['avg_cost_cents']
                new_qty = old_qty + quantity
                # Weighted average cost
                new_avg = (old_qty * old_cost + quantity * price_cents) // new_qty
                cur.execute("""
                    UPDATE positions SET quantity = %s, avg_cost_cents = %s, updated_at = NOW()
                    WHERE user_id = %s AND symbol_id = %s
                """, (new_qty, new_avg, user_id, symbol_id))
                print(f"📈 Updated position: {old_qty} → {new_qty} shares, "
                      f"avg cost ${old_cost/100:.2f} → ${new_avg/100:.2f}")
            else:
                cur.execute("""
                    INSERT INTO positions (user_id, symbol_id, quantity, avg_cost_cents)
                    VALUES (%s, %s, %s, %s)
                """, (user_id, symbol_id, quantity, price_cents))
                print(f"📈 New position: {quantity} shares @ ${price_cents/100:.2f}")

            print(f"💰 Cash deducted: ${total_cost_cents/100:,.2f}  "
                  f"(balance: ${result['balance_cents']/100:,.2f})")

        else:  # sell
            # Check position exists and has enough shares
            cur.execute("""
                SELECT quantity, avg_cost_cents FROM positions
                WHERE user_id = %s AND symbol_id = %s
            """, (user_id, symbol_id))
            existing = cur.fetchone()

            if not existing or existing['quantity'] < quantity:
                raise ValueError(f"Not enough shares! Have {existing['quantity'] if existing else 0}, "
                                 f"trying to sell {quantity}")

            old_qty = existing['quantity']
            avg_cost = existing['avg_cost_cents']
            new_qty = old_qty - quantity

            # Calculate realized P&L
            realized_pnl_cents = (price_cents - avg_cost) * quantity

            if new_qty == 0:
                cur.execute("""
                    DELETE FROM positions WHERE user_id = %s AND symbol_id = %s
                """, (user_id, symbol_id))
            else:
                cur.execute("""
                    UPDATE positions SET quantity = %s, updated_at = NOW()
                    WHERE user_id = %s AND symbol_id = %s
                """, (new_qty, user_id, symbol_id))

            # Add cash
            cur.execute("""
                UPDATE users SET balance_cents = balance_cents + %s
                WHERE id = %s RETURNING balance_cents
            """, (total_cost_cents, user_id))
            result = cur.fetchone()

            pnl_emoji = '🟢' if realized_pnl_cents >= 0 else '🔴'
            print(f"📉 Sold {quantity} shares: {old_qty} → {new_qty} shares")
            print(f"{pnl_emoji} Realized P&L: ${realized_pnl_cents/100:,.2f}")
            print(f"💰 Cash added: ${total_cost_cents/100:,.2f}  "
                  f"(balance: ${result['balance_cents']/100:,.2f})")

        conn.commit()
        print("✅ Transaction committed")

    except Exception as e:
        conn.rollback()
        print(f"❌ Transaction rolled back: {e}")
    finally:
        conn.close()

In [ ]:
# Alice buys 10 more AAPL at $191.50

print("Alice buys 10 more AAPL at $191.50")
print("=" * 50)
process_trade(user_id=1, ticker="AAPL", side="buy", quantity=10, price_cents=19150)

In [ ]:
# Alice sells 20 META at $530.00 (bought at $500)

print("Alice sells 20 META at $530.00")
print("=" * 50)
process_trade(user_id=1, ticker="META", side="sell", quantity=20, price_cents=53000)

In [ ]:
# Bob buys a new stock he didn't own before

print("Bob buys 15 AAPL at $191.00")
print("=" * 50)
process_trade(user_id=2, ticker="AAPL", side="buy", quantity=15, price_cents=19100)

## 💨 Caching Portfolios with Redis

Users check their portfolio **constantly**. Every time they open the app, every time
a price changes, the portfolio recalculates.

We can cache portfolio data in Redis to avoid hitting Postgres on every view.
The cache is invalidated whenever a trade executes.

In [ ]:
r = get_redis()

def get_portfolio(user_id: int) -> list:
    """Get portfolio with Redis cache-aside pattern."""
    cache_key = f"portfolio:{user_id}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        print(f"⚡ Cache HIT for user {user_id}")
        return json.loads(cached)

    print(f"🐘 Cache MISS for user {user_id} — querying Postgres")

    # Fetch from database
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT s.ticker, p.quantity,
               p.avg_cost_cents, s.last_price_cents
        FROM positions p
        JOIN symbols s ON p.symbol_id = s.id
        WHERE p.user_id = %s
        ORDER BY s.ticker
    """, (user_id,))
    positions = [dict(row) for row in cur.fetchall()]
    conn.close()

    # Store in cache with 30 second TTL
    r.setex(cache_key, 30, json.dumps(positions))

    return positions


def invalidate_portfolio_cache(user_id: int):
    """Called after every trade to ensure fresh data."""
    r.delete(f"portfolio:{user_id}")
    print(f"🗑️  Cache invalidated for user {user_id}")


# First call: cache miss
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")
print()

# Second call: cache hit
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")
print()

# After a trade: invalidate
invalidate_portfolio_cache(1)
print()

# Next call: cache miss again
portfolio = get_portfolio(1)
print(f"  → {len(portfolio)} position(s)")

## 📊 Portfolio Summary: Tying It All Together

Let's build a complete portfolio view that shows:
- Each position with current value
- Unrealized P&L per position
- Total portfolio value
- Cash balance

In [ ]:
def display_portfolio(user_id: int):
    """Display a full portfolio summary for a user."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get user info
    cur.execute("SELECT username, balance_cents FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()

    # Get positions with current prices
    cur.execute("""
        SELECT s.ticker, p.quantity,
               p.avg_cost_cents / 100.0 AS avg_cost,
               s.last_price_cents / 100.0 AS market_price,
               p.quantity * s.last_price_cents / 100.0 AS market_value,
               p.quantity * p.avg_cost_cents / 100.0 AS cost_basis,
               (s.last_price_cents - p.avg_cost_cents) * p.quantity / 100.0 AS unrealized_pnl
        FROM positions p
        JOIN symbols s ON p.symbol_id = s.id
        WHERE p.user_id = %s
        ORDER BY p.quantity * s.last_price_cents DESC
    """, (user_id,))
    positions = cur.fetchall()
    conn.close()

    cash = user['balance_cents'] / 100
    total_market_value = sum(float(p['market_value']) for p in positions)
    total_cost = sum(float(p['cost_basis']) for p in positions)
    total_pnl = total_market_value - total_cost
    total_portfolio = total_market_value + cash

    print(f"\n{'='*70}")
    print(f"  📊 {user['username'].upper()}'s Portfolio")
    print(f"{'='*70}")
    print(f"{'Ticker':<8} {'Qty':>5} {'Avg Cost':>10} {'Price':>10} {'Value':>12} {'P&L':>12}")
    print("-" * 70)

    for pos in positions:
        pnl = float(pos['unrealized_pnl'])
        emoji = '🟢' if pnl >= 0 else '🔴'
        print(f"{pos['ticker']:<8} {pos['quantity']:>5} "
              f"${float(pos['avg_cost']):>9,.2f} ${float(pos['market_price']):>9,.2f} "
              f"${float(pos['market_value']):>10,.2f} {emoji}${abs(pnl):>9,.2f}")

    print("-" * 70)
    pnl_emoji = '🟢' if total_pnl >= 0 else '🔴'
    print(f"{'Stocks':<8} {'':>5} {'':>10} {'':>10} ${total_market_value:>10,.2f} "
          f"{pnl_emoji}${abs(total_pnl):>9,.2f}")
    print(f"{'Cash':<8} {'':>5} {'':>10} {'':>10} ${cash:>10,.2f}")
    print(f"{'─'*70}")
    print(f"{'TOTAL':<8} {'':>5} {'':>10} {'':>10} ${total_portfolio:>10,.2f}")
    print()


# View portfolios for Alice and Bob
display_portfolio(1)  # Alice
display_portfolio(2)  # Bob

## 🧹 Cleanup

In [ ]:
# Reset positions and balances to seed-data state
conn = get_db()
cur = conn.cursor()

# Reset Alice: 50 AAPL @ 180, 20 META @ 500
cur.execute("DELETE FROM positions WHERE user_id = 1")
cur.execute("""
    INSERT INTO positions (user_id, symbol_id, quantity, avg_cost_cents) VALUES
    (1, 1, 50, 18000), (1, 3, 20, 50000)
""")
cur.execute("UPDATE users SET balance_cents = 10000000 WHERE id IN (1, 2)")

# Reset Bob: 10 NVDA @ 850
cur.execute("DELETE FROM positions WHERE user_id = 2 AND symbol_id != 7")
cur.execute("UPDATE positions SET quantity = 10, avg_cost_cents = 85000 WHERE user_id = 2 AND symbol_id = 7")

# Clear Redis cache
r = get_redis()
for key in r.keys("portfolio:*"):
    r.delete(key)

print("🧹 Reset positions, balances, and cache to seed-data state")
conn.close()

## 📚 Summary

### Key Takeaways

1. **Positions table** stores one row per (user, symbol) for O(1) portfolio lookups.
2. **Average cost** is recalculated on each buy using a weighted average.
3. **Trades update positions, balance, and orders atomically** in a single DB transaction.
4. **Unrealized P&L** = (current price − avg cost) × quantity (paper gains/losses).
5. **Realized P&L** = (sell price − avg cost) × quantity (actual gains/losses when you sell).
6. **Cache portfolios in Redis** with cache-aside pattern; invalidate on every trade.

### For System Design Interviews

- Explain why you keep a materialized positions table (performance)
- Mention ACID transactions for balance + position updates
- Discuss cache-aside for frequently-viewed portfolio data
- Note that positions are partitioned by `user_id` for single-node queries

### Next Up

In **Notebook 3**, we'll build **real-time market data streaming** — using Kafka
for trade feeds and Redis pub/sub to push live prices to users.